# Justification des Modèles et Spécificités Techniques (EF2)

Ce notebook a pour but de justifier nos choix d'algorithmes, d'expliquer la nécessité (ou non) du scaling selon les modèles, et de désigner notre Modèle Candidat Final.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("customer_churn_business_dataset.csv")

## 1. Pourquoi le Scaling est-il obligatoire pour certains modèles ?

### Les Modèles : Régression Logistique & Deep Learning (MLP)
Ces modèles reposent sur des équations mathématiques où chaque variable est multipliée par un "poids". Si les variables ont des échelles très différentes, les calculs sont faussés (et le gradient descend très lentement).

In [ ]:
print("Comparaison des échelles :")
df[["age", "total_revenue", "monthly_fee"]].describe().loc[["min", "max", "mean"]]

On observe que `total_revenue` peut atteindre des dizaines de milliers, tandis que l'`age` dépasse rarement 100.
**Conséquence** : Dans nos scripts d'entraînement (`train.py`) pour la Régression Logistique et le Deep Learning, nous avons ajouté un `StandardScaler` dans le pipeline.

## 2. L'avantage des modèles basés sur les Arbres

### Les Modèles : Random Forest & XGBoost
Contrairement aux modèles précédents, les arbres de décision scindent les données en cherchant des seuils (ex: si `age` > 30), indépendamment de l'échelle. Ils sont aussi très robustes aux valeurs aberrantes (outliers) et capables de capter des relations non-linéaires complexes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(x="churn", y="total_revenue", data=df, ax=axes[0])
axes[0].set_title("Robustesse aux Outliers")

sns.scatterplot(x="tenure_months", y="total_revenue", hue="churn", data=df, ax=axes[1], alpha=0.6)
axes[1].set_title("Relations Non-Linéaires complexes")
plt.tight_layout()
plt.show()

Le Random Forest et le XGBoost peuvent isoler facilement ces groupes complexes dans le nuage de points sans aucun besoin de normalisation.
**Conséquence** : Dans leurs scripts d'entraînement, nous n'appliquons **pas** de `StandardScaler` (paramètre `passthrough`).

## 3. Choix du Modèle Candidat Final

Suite à nos expérimentations et en regardant le tableau comparatif dans notre Dashboard Streamlit, **notre choix final se porte officiellement sur XGBoost (couplé à SMOTE)**.

**Justifications :**
1. **Performance** : Le XGBoost utilise l'apprentissage séquentiel (Boosting), corrigeant petit à petit ses propres erreurs. C'est redoutable pour les cas limites difficiles à classifier.
2. **Résilience** : Il gère nativement les interactions complexes (voir graphique ci-dessus) et n'a pas besoin de scaling.
3. **Trade-off Recall/Précision** : Associé à SMOTE et un seuil de décision optimisé (0.35), il offre le meilleur équilibre métier pour ne rater aucun départ client imminent tout en gardant un taux acceptable de fausses alertes.